# Day 3: Add search

Welcome to Day 3 of the 7-day AI agents crash course.

We have **downloaded** data (Day 1) and **chunked** it where needed (Day 2). Now we **index** it in a search layer so we can retrieve relevant pieces quickly when users ask questions.

Today:

- **Lexical** (text / keyword) search  
- **Semantic** search with **embeddings**  
- **Hybrid** search (combine both)  

This search stack can later feed your **agent** (Day 4+).

## Setup

```bash
uv add minsearch sentence-transformers numpy tqdm
uv add requests python-frontmatter
uv add --dev jupyter
```

**minsearch** — small in-memory search library ([PyPI](https://pypi.org/project/minsearch/), [source](https://github.com/alexeygrigorev/minsearch)).  
**sentence-transformers** — embedding models (first run downloads weights).

We reuse **`read_repo_data`** (Day 1) and **`sliding_window`** (Day 2) below.

In [1]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"skip {fn}: {e}")
    zf.close()
    return out


def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= n:
            break
    return result

## Prepare chunked Evidently docs

Same pattern as Day 2: sliding window **2000** / step **1000** on `content`.

In [2]:
evidently_docs = read_repo_data("evidentlyai", "docs")

evidently_chunks = []
for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    for chunk in sliding_window(doc_content, 2000, 1000):
        chunk.update(doc_copy)
        evidently_chunks.append(chunk)

len(evidently_docs), len(evidently_chunks)

(95, 576)

## 1. Text search (lexical)

Text search scores documents by **word overlap** with the query (TF‑IDF-style behavior in minsearch). Good for exact terms and product names.

Example question: *“What should be in a test dataset for AI evaluation?”*

In [3]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[],
)

index.fit(evidently_chunks)

In [4]:
query = "What should be in a test dataset for AI evaluation?"
results = index.search(query)
results[:3]

[{'start': 0,
  'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthetic_data_inputs_exa

### DataTalks FAQ (no chunking)

Filter e.g. **data-engineering** entries and index `question` + `content`.

In [5]:
dtc_faq = read_repo_data("DataTalksClub", "faq")

de_dtc_faq = [
    d
    for d in dtc_faq
    if "data-engineering" in (d.get("filename") or "").lower()
]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[],
)

faq_index.fit(de_dtc_faq)
len(de_dtc_faq)

498

In [6]:
q = "What should be in a test dataset for AI evaluation?"
faq_index.search(q, num_results=5)

[{'id': '8bfd724e4f',
  'question': "Compilation Error in test accepted_values_stg_green_tripdata_Payment_type__False___var_payment_type_values_ (models/staging/schema.yml)  'NoneType' object is not iterable",
  'sort_order': 38,
  'content': 'In the macro `test_accepted_values` (found in `tests/generic/builtin.sql`), an error was triggered by the test `accepted_values_stg_green_tripdata_Payment_type__False___var_payment_type_values_` located in `models/staging/schema.yml`.\n\nTo resolve this issue, ensure the following variable is added to your `dbt_project.yml` file:\n\n```yaml\nvars:\n  payment_type_values: [1, 2, 3, 4, 5, 6]\n```',
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/module-4/038_8bfd724e4f_compilation-error-in-test-accepted_values_stg_gree.md'},
 {'id': '9f9a1b9e4f',
  'question': 'Terraform: Teardown of BigQuery Dataset',
  'sort_order': 126,
  'content': "When running `terraform destroy`, the following error can occur:\n\n```\nDo you really want to destr

## 2. Vector (semantic) search

Paraphrases often share **no** keywords — embeddings capture **meaning**.

We use **`multi-qa-distilbert-cos-v1`** (good for QA-style matching). Others: `all-MiniLM-L6-v2` (fast), `all-mpnet-base-v2` (heavier).

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
record = de_dtc_faq[2]
text = record["question"] + " " + record["content"]
v_doc = embedding_model.encode(text)

query = "I just found out about the course. Can I enroll now?"
v_query = embedding_model.encode(query)

similarity = float(v_query.dot(v_doc))
similarity

0.5190931558609009

With **normalized** embeddings, **cosine similarity** matches the **dot product** here.

Build vectors for all FAQ rows, then use **`VectorSearch`**.

In [9]:
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []
for d in tqdm(de_dtc_faq):
    t = d["question"] + " " + d["content"]
    faq_embeddings.append(embedding_model.encode(t))

faq_embeddings = np.array(faq_embeddings)
faq_embeddings.shape

  0%|          | 0/498 [00:00<?, ?it/s]

(498, 768)

In [10]:
from minsearch import VectorSearch

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [11]:
query = "Can I join the course now?"
q = embedding_model.encode(query)
faq_vindex.search(q, num_results=5)

[{'id': '3f1424af17',
  'question': 'Course: Can I still join the course after the start date?',
  'sort_order': 3,
  'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'},
 {'id': '068529125b',
  'question': 'Course - Can I follow the course after it finishes?',
  'sort_order': 8,
  'content': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.',
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/008_068529125b_course-can-i-follow-the-course-after-i

### Evidently chunks — embed `chunk` only

This can take **several minutes** on CPU. Set **`MAX_EVIDENTLY_EMBED`** to a small number for a quick test.

In [ ]:
MAX_EVIDENTLY_EMBED = None  # e.g. 100 for a fast run; None = all chunks

_ev_subset = (
    evidently_chunks[:MAX_EVIDENTLY_EMBED]
    if MAX_EVIDENTLY_EMBED
    else evidently_chunks
)

evidently_embeddings = []
for d in tqdm(_ev_subset):
    evidently_embeddings.append(embedding_model.encode(d["chunk"]))

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, _ev_subset)

  0%|          | 0/576 [00:00<?, ?it/s]

## 3. Hybrid search

- **Text**: fast, strong on keywords.  
- **Vector**: paraphrases and synonyms.  
- **Hybrid**: concatenate top‑k from both and **deduplicate**.

In [ ]:
query = "Can I join the course now?"

text_results = faq_index.search(query, num_results=5)
q_vec = embedding_model.encode(query)
vector_results = faq_vindex.search(q_vec, num_results=5)

final_results = text_results + vector_results
len(text_results), len(vector_results), len(final_results)

## 4. Wrapping in functions

Deduplicate by stable **`id`** (fallback **`filename`**) so the same FAQ row is not returned twice.

In [ ]:
def text_search(query, num_results=5):
    return faq_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=num_results)


def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)
    seen = set()
    combined = []
    for result in text_results + vector_results:
        key = result.get("id") or result.get("filename")
        if key is None:
            key = (result.get("question", "")[:120], result.get("content", "")[:120])
        if key in seen:
            continue
        seen.add(key)
        combined.append(result)
    return combined


hybrid_search("Can I join the course now?", num_results=5)

## 5. Choosing an approach

Start with **lexical** search: fastest to run and debug. Add **vectors** (and then **hybrid**) when queries are paraphrased or vocabulary drifts. Formal **evals** come later in the course.

## Next: conversational agent

Data + search are in place; next steps wire an **agent** on top.

## Homework

- In **`project/`**, index **your** prepared chunks.  
- Compare **text** vs **vector** vs **hybrid**; inspect results manually.  
- Pick a default search mode for the agent (Day 4).
